# Data analysis with pandas

A complete small analysis of the [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/)
dataset: measurements of 344 penguins of three species, collected at Palmer Station, Antarctica,
between 2007 and 2009.

**Question:** can we tell the species apart from their measurements alone?

## Load

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

penguins = pd.read_csv("../data/penguins.csv")
penguins.shape

In [ ]:
penguins.head()

## Inspect

In [ ]:
penguins.dtypes

In [ ]:
penguins.describe()

In [ ]:
penguins.isna().sum()

## Clean

A few rows are missing measurements or sex. There are few enough to drop.

In [ ]:
clean = penguins.dropna().astype({"species": "category", "island": "category", "sex": "category"})
print(f"Kept {len(clean)} of {len(penguins)} rows")

## Summarise

In [ ]:
clean["species"].value_counts()

In [ ]:
pd.crosstab(clean["species"], clean["island"])

Adélie penguins live on all three islands. Gentoo are only on Biscoe and Chinstrap only on Dream.

In [ ]:
measures = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

summary = clean.groupby("species", observed=True)[measures].mean()
summary.style.format("{:.1f}").background_gradient(cmap="Greens")

In [ ]:
clean.pivot_table(index="species", columns="sex", values="body_mass_g", aggfunc="mean", observed=True).round()

## Visualise

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, measure in zip(axes, measures):
    clean.boxplot(column=measure, by="species", ax=ax, grid=False)
    ax.set_title(measure)
    ax.set_xlabel("")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
colours = {"Adelie": "#f97316", "Chinstrap": "#a855f7", "Gentoo": "#0d9488"}

fig, ax = plt.subplots(figsize=(7, 5))
for species, group in clean.groupby("species", observed=True):
    ax.scatter(group["bill_length_mm"], group["bill_depth_mm"], c=colours[species], label=species, alpha=0.7)
ax.set_xlabel("Bill length (mm)")
ax.set_ylabel("Bill depth (mm)")
ax.legend()
plt.show()

## A simple classifier

Bill length and depth separate the species well. Classify each penguin by the species whose
average bill it is closest to.

In [ ]:
centres = clean.groupby("species", observed=True)[["bill_length_mm", "bill_depth_mm"]].mean()

def nearest_species(row):
    distances = ((centres - row[["bill_length_mm", "bill_depth_mm"]].astype(float)) ** 2).sum(axis=1)
    return distances.idxmin()

predicted = clean.apply(nearest_species, axis=1)
accuracy = (predicted == clean["species"]).mean()
print(f"Accuracy: {accuracy:.1%}")

In [ ]:
pd.crosstab(clean["species"], predicted, rownames=["actual"], colnames=["predicted"])

**Answer:** yes. Two bill measurements are enough to classify most penguins correctly.

### Exercise

Add `flipper_length_mm` to the classifier. Does the accuracy improve? (Hint: the measurements
have very different scales, so try standardising them first.)

## Next

Continue with [05-interactive-plots](05-interactive-plots.ipynb).